# Andy's MD Conversion Tool — Full Build Walkthrough

**What this notebook covers:**  
A complete, explained walkthrough of how to build a tool that converts PowerPoint (`.pptx`) and PDF files into structured Markdown (`.md`) — with optional AI-powered image descriptions using Claude.

By the end you'll understand:
- How to extract text and images from PPTX and PDF files with Python
- How to call the Claude API to describe images using vision
- How to structure a multi-file Python project
- How to wrap it all into a desktop GUI app and a shareable `.dmg`

---

## Architecture Overview

The app is built in two layers:

```
┌─────────────────────────────────────┐
│           app.py  (GUI layer)       │
│   CustomTkinter window, buttons,    │
│   file picker, progress bar         │
└────────────────┬────────────────────┘
                 │ calls
┌────────────────▼────────────────────┐
│        converter.py  (logic layer)  │
│  Reads PPTX/PDF → extracts text     │
│  Sends images to Claude Haiku       │
│  Writes .md output files            │
└─────────────────────────────────────┘
```

**This notebook focuses on the logic layer** — the part that actually does the work.  
Every cell below is runnable. At the end, there's a complete CLI version you can run on your own files.

---

## Step 1 — Install Dependencies

We need four key libraries:

| Library | What it does |
|---|---|
| `python-pptx` | Read PowerPoint files |
| `PyMuPDF` (imported as `fitz`) | Read PDF files and extract images |
| `anthropic` | Call the Claude API for image descriptions |
| `Pillow` | Image processing utilities |

In [ ]:
# Run this cell first — it installs everything we need
%pip install python-pptx PyMuPDF anthropic Pillow --quiet

## Step 2 — Set Your API Key

To use Claude for image descriptions, you need a free Anthropic API key.

Get one at: **console.anthropic.com → API Keys**

> **Note:** If you don't have a key, that's fine — all the text extraction cells will still work. Only the image description cells need one.

In [ ]:
import os
import getpass

# This prompts you to enter your key without it showing on screen
# It only lives in memory for this session — it is never saved to disk
ANTHROPIC_API_KEY = getpass.getpass("Enter your Anthropic API key (or press Enter to skip): ")

if ANTHROPIC_API_KEY:
    print("✓ API key set — image descriptions will be enabled")
else:
    print("⚠ No API key — image description cells will be skipped")

---
## Part 1 — Reading PowerPoint Files

We use the `python-pptx` library to open `.pptx` files.  
A PowerPoint file is structured like this:

```
Presentation
 └── Slides  (list of slides)
      └── Shapes  (text boxes, images, tables, etc.)
           └── Text Frames  (paragraphs of text)
```

Let's start by creating a sample `.pptx` file to work with.

In [ ]:
from pptx import Presentation
from pptx.util import Inches, Pt

# Create a simple sample presentation
prs = Presentation()

# Slide 1 — Title slide
slide1 = prs.slides.add_slide(prs.slide_layouts[0])
slide1.shapes.title.text = "Introduction to AI Tools"
slide1.placeholders[1].text = "A practical overview for engineers"

# Slide 2 — Content slide with bullets
slide2 = prs.slides.add_slide(prs.slide_layouts[1])
slide2.shapes.title.text = "What We'll Cover"
tf = slide2.placeholders[1].text_frame
tf.text = "Large Language Models"
tf.add_paragraph().text = "Vision Models"
tf.paragraphs[1].level = 1
tf.add_paragraph().text = "API Integration"
tf.paragraphs[2].level = 1
tf.add_paragraph().text = "Practical Applications"
tf.paragraphs[3].level = 0

# Slide 3 — Another content slide
slide3 = prs.slides.add_slide(prs.slide_layouts[1])
slide3.shapes.title.text = "Key Takeaways"
slide3.placeholders[1].text = "AI is a tool, not a replacement\nStart small and iterate\nAlways review outputs"

prs.save("sample_presentation.pptx")
print("✓ sample_presentation.pptx created")

In [ ]:
# Now let's read that file back
from pptx import Presentation

prs = Presentation("sample_presentation.pptx")

print(f"Number of slides: {len(prs.slides)}")
print()

# Loop through every slide
for i, slide in enumerate(prs.slides, start=1):
    print(f"--- Slide {i} ---")
    
    # Each slide has 'shapes' — text boxes, images, tables, etc.
    for shape in slide.shapes:
        
        # Check if this shape contains text
        if shape.has_text_frame:
            for paragraph in shape.text_frame.paragraphs:
                text = paragraph.text.strip()
                if text:
                    indent = "  " * paragraph.level  # indent based on bullet level
                    print(f"{indent}{text}")
    print()

In [ ]:
# Getting the slide title specifically
# Each slide has a special 'title' shape — let's use it for our Markdown headings

prs = Presentation("sample_presentation.pptx")

for i, slide in enumerate(prs.slides, start=1):
    # slide.shapes.title is the dedicated title placeholder
    slide_title = ""
    if slide.shapes.title and slide.shapes.title.text.strip():
        slide_title = slide.shapes.title.text.strip()
    
    print(f"Slide {i}: '{slide_title}'")

In [ ]:
# Now let's convert a PPTX to Markdown — the core of our tool
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE
from pathlib import Path

def pptx_to_markdown(file_path):
    """Convert a .pptx file to a Markdown string."""
    prs = Presentation(file_path)
    lines = []
    
    # --- Document title ---
    # Try to get the title from file metadata first
    doc_title = Path(file_path).stem  # fallback = filename without extension
    try:
        if prs.core_properties.title:
            doc_title = prs.core_properties.title
    except:
        pass
    
    lines.append(f"# {doc_title}")
    lines.append(f"*Source: {Path(file_path).name}*")
    lines.append("---")
    
    # --- Loop through each slide ---
    for slide_num, slide in enumerate(prs.slides, start=1):
        
        # Get this slide's title
        slide_title = ""
        title_shape = None
        try:
            if slide.shapes.title and slide.shapes.title.text.strip():
                slide_title = slide.shapes.title.text.strip()
                title_shape = slide.shapes.title
        except:
            pass
        
        # Build the slide heading (## Slide N: Title)
        heading = f"## Slide {slide_num}"
        if slide_title:
            heading += f": {slide_title}"
        lines.append(f"\n{heading}\n")
        
        # --- Speaker notes ---
        try:
            notes = slide.notes_slide.notes_text_frame.text.strip()
            if notes:
                lines.append(f"> *Speaker Notes: {notes}*\n")
        except:
            pass
        
        # --- Loop through every shape on this slide ---
        for shape in slide.shapes:
            if shape == title_shape:
                continue  # already used as heading, skip it
            
            # Text shapes
            if shape.has_text_frame:
                for para in shape.text_frame.paragraphs:
                    text = para.text.strip()
                    if not text:
                        continue
                    # Use paragraph indent level for bullet depth
                    if para.level == 0:
                        lines.append(f"\n{text}")
                    else:
                        indent = "  " * (para.level - 1)
                        lines.append(f"{indent}- {text}")
            
            # Table shapes
            elif shape.has_table:
                table = shape.table
                headers = [cell.text.strip() for cell in table.rows[0].cells]
                lines.append("\n| " + " | ".join(headers) + " |")
                lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
                for row in list(table.rows)[1:]:
                    cells = [cell.text.strip() for cell in row.cells]
                    lines.append("| " + " | ".join(cells) + " |")
            
            # Image shapes — placeholder for now
            elif shape.shape_type == MSO_SHAPE_TYPE.PICTURE:
                lines.append("\n> **[Image]** *Description coming in Part 3*\n")
    
    return "\n".join(lines)


# Run it!
markdown_output = pptx_to_markdown("sample_presentation.pptx")
print(markdown_output)

---
## Part 2 — Reading PDF Files

We use **PyMuPDF** (imported as `fitz`) to read PDFs.  
It's one of the most powerful PDF libraries available — it handles text, images, and layout.

A PDF is structured like this:
```
Document
 └── Pages
      └── Blocks  (text blocks, image blocks)
           └── Lines → Spans → Text
```

In [ ]:
# First, let's create a sample PDF to work with
# We'll use reportlab — install it if needed
%pip install reportlab --quiet

from reportlab.lib.pagesizes import letter
from reportlab.platypus import SimpleDocTemplate, Paragraph, Spacer
from reportlab.lib.styles import getSampleStyleSheet

doc = SimpleDocTemplate("sample_document.pdf", pagesize=letter)
styles = getSampleStyleSheet()
story = []

story.append(Paragraph("Introduction to AI Engineering", styles['Title']))
story.append(Spacer(1, 20))
story.append(Paragraph("Chapter 1: What is a Large Language Model?", styles['Heading1']))
story.append(Spacer(1, 10))
story.append(Paragraph(
    "A large language model (LLM) is a type of machine learning model trained on "
    "vast amounts of text data. It learns statistical patterns in language and can "
    "generate coherent, contextually relevant text based on a given prompt.",
    styles['Normal']
))
story.append(Spacer(1, 10))
story.append(Paragraph("Chapter 2: The Claude API", styles['Heading1']))
story.append(Spacer(1, 10))
story.append(Paragraph(
    "Anthropic's Claude API gives developers access to Claude models via simple "
    "HTTP requests. You send a message, and Claude sends one back. This tool uses "
    "the Claude Haiku model — Anthropic's fastest and most cost-effective option.",
    styles['Normal']
))

doc.build(story)
print("✓ sample_document.pdf created")

In [ ]:
import fitz  # PyMuPDF — the import name is 'fitz' even though we installed 'PyMuPDF'

doc = fitz.open("sample_document.pdf")

print(f"Number of pages: {len(doc)}")
print(f"Document title: {doc.metadata.get('title', '(none in metadata)')}")
print()

# Loop through pages
for page_num, page in enumerate(doc, start=1):
    print(f"--- Page {page_num} ---")
    
    # Simple text extraction
    text = page.get_text()
    print(text[:300])  # Print first 300 characters
    print()

doc.close()

In [ ]:
# For our converter we use 'dict' mode — it gives us structured blocks
# This lets us handle text and images separately

doc = fitz.open("sample_document.pdf")
page = doc[0]  # First page

# get_text("dict") returns a dictionary with 'blocks'
# Each block is either type 0 (text) or type 1 (image)
page_dict = page.get_text("dict", sort=True)

for block in page_dict["blocks"]:
    block_type = block["type"]
    
    if block_type == 0:  # Text block
        # A block contains lines, lines contain spans, spans contain text
        for line in block["lines"]:
            line_text = " ".join(span["text"] for span in line["spans"]).strip()
            if line_text:
                print(f"TEXT: {line_text}")
    
    elif block_type == 1:  # Image block
        print(f"IMAGE: xref={block.get('image')}")

doc.close()

In [ ]:
# Now let's convert a PDF to Markdown
import fitz
from pathlib import Path

def pdf_to_markdown(file_path):
    """Convert a PDF file to a Markdown string."""
    doc = fitz.open(file_path)
    lines = []
    
    # --- Document title from metadata ---
    title = Path(file_path).stem  # fallback = filename
    try:
        if doc.metadata.get('title') and doc.metadata['title'].strip():
            title = doc.metadata['title'].strip()
    except:
        pass
    
    lines.append(f"# {title}")
    lines.append(f"*Source: {Path(file_path).name}*")
    lines.append("---")
    
    # --- Loop through pages ---
    for page_num, page in enumerate(doc, start=1):
        lines.append(f"\n## Page {page_num}\n")
        
        seen_xrefs = set()  # track which images we've already processed
        blocks = page.get_text("dict", sort=True)["blocks"]
        
        for block in blocks:
            
            if block["type"] == 0:  # Text block
                for line in block["lines"]:
                    text = " ".join(s["text"] for s in line["spans"]).strip()
                    if text:
                        lines.append(text)
            
            elif block["type"] == 1:  # Inline image block
                xref = block.get("image")
                if xref and xref not in seen_xrefs:
                    seen_xrefs.add(xref)
                    lines.append("\n> **[Image]** *Description coming in Part 3*\n")
        
        # Catch any images not in the blocks (common in some PDFs)
        for img_info in page.get_images(full=True):
            xref = img_info[0]
            if xref not in seen_xrefs:
                seen_xrefs.add(xref)
                lines.append("\n> **[Image]** *Description coming in Part 3*\n")
    
    doc.close()
    return "\n".join(lines)


# Run it!
markdown_output = pdf_to_markdown("sample_document.pdf")
print(markdown_output)

---
## Part 3 — AI Image Descriptions with Claude

This is where the magic happens.  
Instead of silently skipping images, we send each one to Claude Haiku — Anthropic's fastest, cheapest model — and ask it to:

- Describe what the image shows
- Extract any text visible in the image
- Describe charts, graphs, or diagrams in detail

The result gets embedded in the Markdown as a readable description.

### How the Claude API works

```
You                          Claude API
 │                               │
 │── POST /messages ────────────>│
 │   { model, messages: [        │
 │     { role: "user",           │
 │       content: [              │
 │         { type: "image" },    │
 │         { type: "text" }      │
 │       ]}                      │
 │   ]}                          │
 │                               │
 │<── { content: [{text: "..."}]} │
```

Images must be sent as **base64-encoded strings** — that's just a way of turning binary image data into plain text that can travel over HTTP.

In [ ]:
import anthropic
import base64

# Only run this if we have an API key
if not ANTHROPIC_API_KEY:
    print("⚠ Skipping — no API key provided")
else:
    # Create the client — this is our connection to Claude
    client = anthropic.Anthropic(api_key=ANTHROPIC_API_KEY)
    
    # Simple text-only test first
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",   # fastest + cheapest model
        max_tokens=100,
        messages=[
            {"role": "user", "content": "Say hello in one sentence."}
        ]
    )
    
    print("Claude says:", response.content[0].text)

In [ ]:
# Create a test image with text in it (to demonstrate image description)
from PIL import Image, ImageDraw, ImageFont
import io

# Create a simple chart-like image
img = Image.new('RGB', (400, 250), color='white')
draw = ImageDraw.Draw(img)

# Title
draw.text((20, 15), "Q4 Revenue by Region", fill='black')
draw.text((20, 35), "(in millions USD)", fill='gray')

# Simple bar chart
bars = [("North", 82, "#4472C4"), ("South", 65, "#ED7D31"),
        ("East", 91, "#A9D18E"), ("West", 74, "#FF0000")]

for i, (label, value, color) in enumerate(bars):
    x = 40 + i * 90
    bar_height = int(value * 1.5)
    draw.rectangle([x, 220 - bar_height, x + 60, 220], fill=color)
    draw.text((x + 10, 225), label, fill='black')
    draw.text((x + 15, 220 - bar_height - 18), f"${value}M", fill='black')

# Save to bytes (we'll send this to Claude)
img_bytes = io.BytesIO()
img.save(img_bytes, format='PNG')
img_bytes = img_bytes.getvalue()

# Also save to disk so you can see it
img.save("test_chart.png")

print(f"✓ Test image created — {len(img_bytes):,} bytes")

# Display it in the notebook
img

In [ ]:
# Now send that image to Claude for a description
import base64

if not ANTHROPIC_API_KEY:
    print("⚠ Skipping — no API key provided")
else:
    # Step 1: Convert image bytes to base64 string
    # (base64 turns binary data into plain text so it can be sent in JSON)
    b64_image = base64.standard_b64encode(img_bytes).decode('utf-8')
    
    # Step 2: Send to Claude with both an image and a text instruction
    response = client.messages.create(
        model="claude-haiku-4-5-20251001",
        max_tokens=400,
        messages=[{
            "role": "user",
            "content": [
                {
                    "type": "image",
                    "source": {
                        "type": "base64",
                        "media_type": "image/png",
                        "data": b64_image     # <-- the image, as base64 text
                    }
                },
                {
                    "type": "text",
                    "text": (
                        "Describe this image concisely. "
                        "Extract ALL visible text verbatim. "
                        "For charts and graphs, describe the data shown."
                    )
                }
            ]
        }]
    )
    
    description = response.content[0].text
    print("Claude's description:")
    print("-" * 40)
    print(description)

In [ ]:
# Let's turn this into a reusable function
import base64
import anthropic

def describe_image(image_bytes, context="", api_key=None):
    """
    Send an image to Claude Haiku and get a description back.
    
    Args:
        image_bytes: Raw image data as bytes
        context: A hint to Claude about where this image came from
        api_key: Your Anthropic API key
    
    Returns:
        A string description of the image
    """
    if not api_key:
        return "[Image — description disabled, no API key]"
    
    if not image_bytes or len(image_bytes) < 2000:  # skip tiny/blank images
        return None
    
    # Detect the image format from its first bytes (called 'magic bytes')
    if image_bytes[:8] == b'\x89PNG\r\n\x1a\n':
        media_type = "image/png"
    elif image_bytes[:2] == b'\xff\xd8':
        media_type = "image/jpeg"
    else:
        media_type = "image/png"  # default
    
    b64 = base64.standard_b64encode(image_bytes).decode('utf-8')
    
    client = anthropic.Anthropic(api_key=api_key)
    
    try:
        response = client.messages.create(
            model="claude-haiku-4-5-20251001",
            max_tokens=400,
            messages=[{
                "role": "user",
                "content": [
                    {
                        "type": "image",
                        "source": {
                            "type": "base64",
                            "media_type": media_type,
                            "data": b64
                        }
                    },
                    {
                        "type": "text",
                        "text": (
                            f"Context: {context}\n\n"
                            "Describe this image concisely. "
                            "Extract ALL visible text verbatim. "
                            "For charts, graphs, or tables, describe the data shown. "
                            "For diagrams, describe the structure and labels."
                        )
                    }
                ]
            }]
        )
        return response.content[0].text.strip()
    
    except Exception as e:
        return f"[Image description unavailable: {e}]"


# Test it
if ANTHROPIC_API_KEY:
    result = describe_image(img_bytes, context="Slide 3 of 'Q4 Results'", api_key=ANTHROPIC_API_KEY)
    print(result)
else:
    print(describe_image(img_bytes, api_key=None))

---
## Part 4 — Cost Estimation

Before processing a batch of files, the app estimates how much the API calls will cost.

Claude Haiku pricing (approximate):
- **Input tokens:** $0.80 per million tokens
- **Output tokens:** $4.00 per million tokens
- **Images:** Each image costs roughly ~1,500 input tokens

For a 20-slide deck with 5 images, the cost is typically **less than $0.01**.

In [ ]:
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE
import fitz

# Pricing constants
INPUT_COST_PER_MTOK  = 0.80   # $0.80 per million input tokens
OUTPUT_COST_PER_MTOK = 4.00   # $4.00 per million output tokens
TOKENS_PER_IMAGE_IN  = 1500   # estimated tokens per image (input)
TOKENS_PER_IMAGE_OUT = 200    # estimated tokens per image (output = description)


def count_images_in_pptx(path):
    """Count the number of images in a PPTX file."""
    prs = Presentation(path)
    return sum(
        1 for slide in prs.slides
        for shape in slide.shapes
        if shape.shape_type == MSO_SHAPE_TYPE.PICTURE
    )

def count_images_in_pdf(path):
    """Count the number of images in a PDF file."""
    doc = fitz.open(path)
    count = sum(len(page.get_images()) for page in doc)
    doc.close()
    return count

def estimate_cost(image_count):
    """Estimate API cost for describing N images."""
    input_tokens  = image_count * TOKENS_PER_IMAGE_IN
    output_tokens = image_count * TOKENS_PER_IMAGE_OUT
    cost = (input_tokens  * INPUT_COST_PER_MTOK  / 1_000_000 +
            output_tokens * OUTPUT_COST_PER_MTOK / 1_000_000)
    return cost


# Test it on our sample files
pptx_images = count_images_in_pptx("sample_presentation.pptx")
pdf_images  = count_images_in_pdf("sample_document.pdf")
total       = pptx_images + pdf_images
cost        = estimate_cost(total)

print(f"PPTX images:  {pptx_images}")
print(f"PDF images:   {pdf_images}")
print(f"Total images: {total}")
print(f"Est. cost:    ${cost:.4f}")

---
## Part 5 — The Complete Converter (CLI Version)

Now let's put everything together into a complete, runnable converter.  
This is the `converter.py` file from the actual app — simplified slightly for clarity.

This version runs from the command line (or right here in the notebook) without needing a GUI.

In [ ]:
import base64
import anthropic
import fitz
from pptx import Presentation
from pptx.enum.shapes import MSO_SHAPE_TYPE
from pathlib import Path

# ── Constants ────────────────────────────────────────────────────────────────
HAIKU_MODEL          = "claude-haiku-4-5-20251001"
INPUT_COST_PER_MTOK  = 0.80
OUTPUT_COST_PER_MTOK = 4.00
EST_INPUT_TOKENS     = 1500
EST_OUTPUT_TOKENS    = 200
MIN_IMAGE_BYTES      = 2000   # skip images smaller than this (icons, decorations)


# ── Main converter class ─────────────────────────────────────────────────────
class FileConverter:
    """
    Converts .pptx and .pdf files to Markdown.
    Optionally uses Claude Haiku to describe images.
    """
    
    def __init__(self, api_key=None, images_enabled=True):
        self.images_enabled = images_enabled
        # Only create the API client if image descriptions are on
        self.client = anthropic.Anthropic(api_key=api_key) if (images_enabled and api_key) else None
    
    # ── Public methods ───────────────────────────────────────────────────────
    
    def estimate_images(self, file_paths):
        """Count total images across a list of files."""
        total = 0
        for fp in file_paths:
            ext = Path(fp).suffix.lower()
            if ext in ('.pptx', '.ppt'):
                total += self._count_pptx_images(fp)
            elif ext == '.pdf':
                total += self._count_pdf_images(fp)
        return total
    
    def estimate_cost(self, image_count):
        """Estimate API cost in USD for describing N images."""
        return (image_count * EST_INPUT_TOKENS  * INPUT_COST_PER_MTOK  / 1_000_000 +
                image_count * EST_OUTPUT_TOKENS * OUTPUT_COST_PER_MTOK / 1_000_000)
    
    def convert_file(self, file_path, output_folder):
        """
        Convert a single file and save the .md output.
        Returns (success: bool, output_path: str, message: str)
        """
        ext = Path(file_path).suffix.lower()
        output_path = Path(output_folder) / f"{Path(file_path).stem}.md"
        
        if ext in ('.pptx', '.ppt'):
            markdown = self._convert_pptx(file_path)
        elif ext == '.pdf':
            markdown = self._convert_pdf(file_path)
        else:
            return False, None, f"Unsupported format: {ext}"
        
        output_path.write_text(markdown, encoding='utf-8')
        return True, str(output_path), "Success"
    
    # ── Image counting ───────────────────────────────────────────────────────
    
    def _count_pptx_images(self, path):
        prs = Presentation(path)
        return sum(
            1 for slide in prs.slides
            for shape in slide.shapes
            if shape.shape_type == MSO_SHAPE_TYPE.PICTURE
        )
    
    def _count_pdf_images(self, path):
        doc = fitz.open(path)
        count = sum(len(page.get_images()) for page in doc)
        doc.close()
        return count
    
    # ── Image description (Claude Haiku) ─────────────────────────────────────
    
    def _describe_image(self, image_bytes, context=""):
        """Ask Claude to describe an image. Returns a string."""
        
        # If image descriptions are turned off, return a placeholder
        if not self.images_enabled or not self.client:
            return "[Image — description disabled]"
        
        # Skip images that are too small to be meaningful
        if not image_bytes or len(image_bytes) < MIN_IMAGE_BYTES:
            return None
        
        # Detect image format from magic bytes
        if image_bytes[:8] == b'\x89PNG\r\n\x1a\n':
            media_type = "image/png"
        elif image_bytes[:2] == b'\xff\xd8':
            media_type = "image/jpeg"
        elif image_bytes[8:12] == b'WEBP':
            media_type = "image/webp"
        else:
            media_type = "image/png"
        
        b64 = base64.standard_b64encode(image_bytes).decode('utf-8')
        
        try:
            response = self.client.messages.create(
                model=HAIKU_MODEL,
                max_tokens=400,
                messages=[{
                    "role": "user",
                    "content": [
                        {
                            "type": "image",
                            "source": {
                                "type": "base64",
                                "media_type": media_type,
                                "data": b64
                            }
                        },
                        {
                            "type": "text",
                            "text": (
                                f"Context: {context}\n\n"
                                "Describe this image concisely. "
                                "Extract ALL visible text verbatim. "
                                "For charts/graphs/tables, describe the data shown."
                            )
                        }
                    ]
                }]
            )
            return response.content[0].text.strip()
        
        except Exception as e:
            return f"[Image description unavailable: {e}]"
    
    # ── PPTX conversion ──────────────────────────────────────────────────────
    
    def _convert_pptx(self, file_path):
        prs = Presentation(file_path)
        lines = []
        
        # Document title
        title = Path(file_path).stem
        try:
            if prs.core_properties.title and prs.core_properties.title.strip():
                title = prs.core_properties.title.strip()
        except:
            pass
        
        lines.append(f"# {title}")
        lines.append(f"*Source: {Path(file_path).name}*")
        lines.append("---")
        
        for slide_num, slide in enumerate(prs.slides, start=1):
            print(f"  Processing slide {slide_num}/{len(prs.slides)}...")
            
            # Slide title
            slide_title = ""
            title_shape = None
            try:
                if slide.shapes.title and slide.shapes.title.text.strip():
                    slide_title = slide.shapes.title.text.strip()
                    title_shape = slide.shapes.title
            except:
                pass
            
            heading = f"## Slide {slide_num}"
            if slide_title:
                heading += f": {slide_title}"
            lines.append(f"\n{heading}\n")
            
            # Speaker notes
            try:
                notes = slide.notes_slide.notes_text_frame.text.strip()
                if notes:
                    lines.append(f"> *Speaker Notes: {notes}*\n")
            except:
                pass
            
            # Shapes
            for shape in slide.shapes:
                if shape == title_shape:
                    continue
                
                if shape.has_text_frame:
                    for para in shape.text_frame.paragraphs:
                        text = para.text.strip()
                        if not text:
                            continue
                        if para.level == 0:
                            lines.append(f"\n{text}")
                        else:
                            lines.append("  " * (para.level - 1) + f"- {text}")
                
                elif shape.has_table:
                    headers = [c.text.strip() for c in shape.table.rows[0].cells]
                    lines.append("\n| " + " | ".join(headers) + " |")
                    lines.append("| " + " | ".join(["---"] * len(headers)) + " |")
                    for row in list(shape.table.rows)[1:]:
                        lines.append("| " + " | ".join(c.text.strip() for c in row.cells) + " |")
                
                elif shape.shape_type == MSO_SHAPE_TYPE.PICTURE:
                    desc = self._describe_image(
                        shape.image.blob,
                        context=f"Slide {slide_num} of '{title}'"
                    )
                    if desc:
                        lines.append(f"\n> **[Image]** {desc}\n")
        
        return "\n".join(lines)
    
    # ── PDF conversion ───────────────────────────────────────────────────────
    
    def _convert_pdf(self, file_path):
        doc = fitz.open(file_path)
        lines = []
        
        # Document title
        title = Path(file_path).stem
        try:
            if doc.metadata.get('title') and doc.metadata['title'].strip():
                title = doc.metadata['title'].strip()
        except:
            pass
        
        lines.append(f"# {title}")
        lines.append(f"*Source: {Path(file_path).name}*")
        lines.append("---")
        
        for page_num, page in enumerate(doc, start=1):
            print(f"  Processing page {page_num}/{len(doc)}...")
            lines.append(f"\n## Page {page_num}\n")
            
            seen_xrefs = set()
            blocks = page.get_text("dict", sort=True)["blocks"]
            
            for block in blocks:
                if block["type"] == 0:  # text
                    for line in block["lines"]:
                        text = " ".join(s["text"] for s in line["spans"]).strip()
                        if text:
                            lines.append(text)
                
                elif block["type"] == 1:  # inline image
                    xref = block.get("image")
                    if xref and xref not in seen_xrefs:
                        seen_xrefs.add(xref)
                        try:
                            img_bytes = doc.extract_image(xref)["image"]
                            if len(img_bytes) >= MIN_IMAGE_BYTES:
                                desc = self._describe_image(img_bytes, f"Page {page_num} of '{title}'")
                                if desc:
                                    lines.append(f"\n> **[Image]** {desc}\n")
                        except:
                            pass
            
            # Catch images not surfaced in blocks
            for img_info in page.get_images(full=True):
                xref = img_info[0]
                if xref not in seen_xrefs:
                    seen_xrefs.add(xref)
                    try:
                        img_bytes = doc.extract_image(xref)["image"]
                        if len(img_bytes) >= MIN_IMAGE_BYTES:
                            desc = self._describe_image(img_bytes, f"Page {page_num} of '{title}'")
                            if desc:
                                lines.append(f"\n> **[Image]** {desc}\n")
                    except:
                        pass
        
        doc.close()
        return "\n".join(lines)


print("✓ FileConverter class defined")

---
## Part 6 — Run the Converter

Now let's use the `FileConverter` class to convert our sample files.

In [ ]:
import os
from pathlib import Path

# Files to convert
files_to_convert = [
    "sample_presentation.pptx",
    "sample_document.pdf"
]

# Where to save the output
output_folder = "markdown_output"
Path(output_folder).mkdir(exist_ok=True)

# Create the converter
# Pass your API key to enable image descriptions, or leave it out to skip them
converter = FileConverter(
    api_key=ANTHROPIC_API_KEY if ANTHROPIC_API_KEY else None,
    images_enabled=bool(ANTHROPIC_API_KEY)
)

# Estimate cost before running
image_count = converter.estimate_images(files_to_convert)
cost = converter.estimate_cost(image_count)
print(f"Images found: {image_count}")
print(f"Estimated cost: ${cost:.4f}")
print()

# Convert each file
for file_path in files_to_convert:
    print(f"Converting: {file_path}")
    success, output_path, message = converter.convert_file(file_path, output_folder)
    
    if success:
        print(f"  ✓ Saved to: {output_path}")
    else:
        print(f"  ✗ Failed: {message}")
    print()

In [ ]:
# Let's read back one of the output files to see the result
output_file = Path("markdown_output/sample_presentation.md")

if output_file.exists():
    print(output_file.read_text())
else:
    print("File not found — run the cell above first")

In [ ]:
# And the PDF output
output_file = Path("markdown_output/sample_document.md")

if output_file.exists():
    print(output_file.read_text())
else:
    print("File not found — run the conversion cell above first")

---
## Part 7 — The GUI Layer (Overview)

The actual desktop app wraps everything above in a GUI built with **CustomTkinter** — a modern-looking skin on top of Python's built-in `tkinter` UI library.

The GUI can't run inside a notebook (it needs a real desktop window), but here's how it's structured:

```
ConverterApp  (main window)
│
├── WelcomeDialog       shown on first run
│    ├── Explains the app
│    ├── Explains the API key & what it unlocks
│    └── Enable / disable image descriptions
│
├── Left panel — File list
│    ├── Scrollable list of selected files
│    └── Add / Remove / Clear buttons
│
├── Right panel — Settings
│    ├── AI Image toggle (on/off switch)
│    ├── API Key field (saved to Mac Keychain, not plain text)
│    ├── Output folder picker
│    ├── Existing file behaviour (overwrite / skip / ask)
│    ├── Analyze button  → counts images, estimates cost
│    └── Convert button  → runs the FileConverter
│
└── Footer
     ├── Progress bar
     └── Status label
```

### Key GUI patterns used

**Threading** — the conversion runs on a background thread so the UI doesn't freeze:
```python
import threading
threading.Thread(target=self._run_conversion, daemon=True).start()
```

**Callbacks** — the converter reports progress back to the GUI:
```python
conv = FileConverter(api_key, log_callback=lambda msg: status_label.configure(text=msg))
```

**Mac Keychain** — in the desktop app, the API key is stored securely using the `keyring` library. When the user clicks **Save Key**, this runs:
```python
import keyring
keyring.set_password("AndysMDConversionTool", "anthropic_api_key", key)
```
And on every launch, this loads it back automatically:
```python
key = keyring.get_password("AndysMDConversionTool", "anthropic_api_key")
```
The key is stored in the Mac's system Keychain — the same place Safari and iCloud store passwords. It never touches a plain text file.

> **⚠ Important — this notebook does NOT use Keychain.**  
> In this notebook, the API key is entered via `getpass` at the top and lives **in memory only** for this session. It is never saved to disk. There is no Keychain integration here — that only exists in the desktop app (`app.py`). Each person who installs the desktop app saves their own key to their own Mac's Keychain.

---
## Part 8 — Packaging as a Desktop App

> **⚠ This section cannot be run in a notebook.**  
> Building a `.app` and `.dmg` requires Mac-specific command-line tools (`PyInstaller` and `create-dmg`) that need access to your desktop environment. These commands must be run in a **Mac Terminal**, not in Jupyter or Google Colab.

---

### Step 1 — Install the tools (Terminal)

```bash
pip install pyinstaller
brew install create-dmg
```

### Step 2 — Build the .app bundle (Terminal)

PyInstaller reads your Python scripts and bundles them — along with all dependencies and a copy of Python itself — into a self-contained `.app` folder that anyone can double-click, even if they've never installed Python.

```bash
pyinstaller \
  --name "MyApp" \
  --windowed \
  --onedir \
  --collect-all customtkinter \
  --collect-all anthropic \
  --collect-all keyring \
  app.py
```

| Flag | What it does |
|---|---|
| `--windowed` | Hides the Terminal window when the app launches |
| `--onedir` | Puts everything in one folder (faster launch than `--onefile`) |
| `--collect-all` | Makes sure all files for that library are included |

The result is `dist/MyApp.app`.

### Step 3 — Create the .dmg installer (Terminal)

A `.dmg` is a standard Mac disk image — the classic "drag to Applications" installer experience.

```bash
create-dmg \
  --volname "My App" \
  --window-size 600 400 \
  --app-drop-link 450 185 \
  "MyApp.dmg" \
  "dist/MyApp.app"
```

The result is a single `MyApp.dmg` file you can share with anyone.

### Why can't this run in a notebook?

Three reasons:
1. `PyInstaller` needs to introspect and copy your entire Python environment — it has to run on the same Mac the app will run on
2. `--windowed` builds a Mac `.app` bundle, which is a macOS-specific format
3. `create-dmg` uses macOS Finder scripting (`AppleScript`) to lay out the installer window — this only works on a real Mac desktop

---

## Summary

| Component | Library | Runs in notebook? | What it does |
|---|---|---|---|
| Read PPTX | `python-pptx` | ✅ Yes | Open slides, extract text/images/tables |
| Read PDF | `PyMuPDF` (fitz) | ✅ Yes | Open pages, extract text blocks and images |
| Describe images | `anthropic` + Claude Haiku | ✅ Yes | Vision API — describes images as text |
| GUI | `customtkinter` | ❌ No | Desktop window, buttons, progress bar |
| Secure key storage | `keyring` (Mac Keychain) | ❌ No | Saves API key to Mac Keychain |
| Packaging | `PyInstaller` + `create-dmg` | ❌ No (Mac Terminal only) | Bundles app into a .app / .dmg |

The full source code is in two files:
- **`converter.py`** — all the logic (everything covered in Parts 1–6 of this notebook)
- **`app.py`** — the GUI wrapper, Keychain integration, and packaging entry point

In [ ]:
# Optional cleanup — remove sample files created during this notebook
import os, shutil

for f in ["sample_presentation.pptx", "sample_document.pdf", "test_chart.png"]:
    if os.path.exists(f):
        os.remove(f)

if os.path.exists("markdown_output"):
    shutil.rmtree("markdown_output")

print("✓ Cleaned up sample files")